# 01 — HDDM model estimation

Fit one candidate hierarchical DDM, inspect traces, and export group-level CSV summaries plus a local InferenceData (`.nc`) file.

## Prerequisites

1. Use **Python 3.8** with **PyMC 2.3.8** and **HDDM 0.9.x** (see `../requirements.txt`). HDDM does **not** run on Python 3.10+.
2. Install HDDM, then apply the multi-chain patch **once**:
   ```bash
   python ../scripts/patch_kabuki_multichain.py
   ```
3. Prefer opening this notebook under **`hddm_analysis/notebooks/`**. Package paths are resolved via `_resolve_hddm_root()` (marker: `data/trials_hddm_ready.tsv`), so outputs stay under `hddm_analysis/` even if the kernel cwd is the workspace root.

**Runtime:** MCMC may take many hours per model (4 chains × 10 000 iterations).


In [ ]:
import importlib.util
import os
import time
from pathlib import Path

import pickle

import arviz as az
import hddm
import numpy as np
import pandas as pd

def _resolve_hddm_root() -> Path:
    """Locate hddm_analysis/ regardless of notebook cwd."""
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "data" / "trials_hddm_ready.tsv").is_file():
            return candidate
        nested = candidate / "hddm_analysis"
        if (nested / "data" / "trials_hddm_ready.tsv").is_file():
            return nested
    if start.name == "notebooks" and (start.parent / "data").is_dir():
        return start.parent
    raise FileNotFoundError(
        "Could not locate hddm_analysis package root "
        "(expected data/trials_hddm_ready.tsv). "
        f"Started search from: {start}"
    )


ROOT = str(_resolve_hddm_root())
DATA_DIR = os.path.join(ROOT, "data")
RESULTS_DIR = os.path.join(ROOT, "results")
MODEL_DIR = os.path.join(ROOT, "models", "transition_prob_duration_10000samples")
FIG_DIR = os.path.join(ROOT, "figures")
for _d in (DATA_DIR, RESULTS_DIR, MODEL_DIR, FIG_DIR):
    os.makedirs(_d, exist_ok=True)


def flip_errors(data: pd.DataFrame) -> pd.DataFrame:
    if np.any(data["rt"] < 0):
        return data
    out = data.copy()
    out.loc[out["response"] != 1, "rt"] = -out.loc[out["response"] != 1, "rt"]
    return out


# Load HDDM-ready data from bundled TSV.
# Exclude anticipatory / invalid RTs (absolute RT ≤ 150 ms) before fitting.
MIN_RT_MS = 150

if "hddm_data" not in globals():
    trials = pd.read_csv(os.path.join(DATA_DIR, "trials_hddm_ready.tsv"), sep="\t")
    n_raw = len(trials)
    trials = trials.loc[trials["rt_ms"] > MIN_RT_MS].copy()
    n_drop = n_raw - len(trials)
    hddm_data = trials.copy()
    hddm_data["response"] = hddm_data["acc"].astype(int)
    hddm_data["rt"] = hddm_data["rt_ms"] / 1000.0
    hddm_data = hddm_data[
        ["subj_idx", "transition", "prob", "long_short", "response", "rt"]
    ]
    hddm_data = flip_errors(hddm_data)
    print(
        f"Loaded hddm_data: {hddm_data.shape} "
        f"(dropped {n_drop} trials with rt_ms ≤ {MIN_RT_MS})"
    )
else:
    print("Using existing hddm_data:", hddm_data.shape)


HDDM: pytorch module seems missing. No LAN functionality can be loaded.
It seems that you do not have pytorch installed. You cannot use the network_inspector module.
It seems that you do not have pytorch installed.The HDDMnn, HDDMnnRegressor, HDDMnnStimCoding, HDDMnnRL and HDDMnnRLRegressorclasses will not work
It seems that you do not have pytorch installed.The HDDMnn, HDDMnnRegressor, HDDMnnStimCoding, HDDMnnRL and HDDMnnRLRegressorclasses will not work
It seems that you do not have pytorch installed.The HDDMnn, HDDMnnRegressor and HDDMnnStimCodingclasses will not work
It seems that you do not have pytorch installed.The HDDMnn, HDDMnnRL, HDDMnnRegressor, HDDMnnStimCoding, HDDMnnRL and HDDMnnRLRegressorclasses will not work
It seems that you do not have pytorch installed.The HDDMnn, HDDMnnRegressor, HDDMnnStimCoding, HDDMnnRL and HDDMnnRLRegressorclasses will not work
Loaded hddm_data: (5160, 6)


## Model configuration

Set **`ACTIVE_MODEL_STEM`** to the candidate you want to fit. The stem is derived automatically from `depends_on` / `include` (naming rule: `hddm_{dep}_{inc}` with `v,a,t,z` order).

**Default:** `hddm_va_vat` — manuscript winning model (`v` and `a` vary by condition; `t` is a group scalar).

Set **`TEST_RUN = True`** for a quick smoke test (`N_ITER` / `BURN` as you choose). Outputs go to `models/.../_test_runs/` and **never overwrite** production CSV / `.nc` files.

All 25 candidates used in model comparison are listed in **`MODEL_CANDIDATES`** in the next cell.

In [2]:
CONDITION = ["transition", "prob", "long_short"]
PARAM_ORDER = ["v", "a", "t", "z"]

# --- edit this line to fit a different candidate ---
ACTIVE_MODEL_STEM = "hddm_va_vat"

# True = smoke test; writes to MODEL_DIR/_test_runs/ (does not overwrite production files)
TEST_RUN = True

# MCMC settings (recorded in model_info.csv)
N_CHAINS = 4
N_ITER = 100 if TEST_RUN else 10000
BURN = 20 if TEST_RUN else 2000
P_OUTLIER = 0.05

# Set True to drop *_subj* nodes before writing .nc
EXPORT_GROUP_LEVEL_ONLY = False

ALL_MODEL_STEMS = [
    "hddm_a_a", "hddm_a_at", "hddm_a_atz", "hddm_a_az", "hddm_a_va", "hddm_a_vat",
    "hddm_a_vatz", "hddm_a_vaz", "hddm_v_v", "hddm_v_va", "hddm_v_vat", "hddm_v_vatz",
    "hddm_v_vaz", "hddm_v_vt", "hddm_v_vtz", "hddm_v_vz", "hddm_va_va", "hddm_va_vat", "hddm_va_vatz",
    "hddm_va_vaz", "hddm_vat_vat", "hddm_vat_vatz", "hddm_vaz_vaz", "hddm_vaz_vatz",
    "hddm_vatz_vatz",
]


def model_file_stem(depends_on: dict, include: list) -> str:
    dep = "".join(p for p in PARAM_ORDER if p in depends_on)
    inc = "".join(p for p in PARAM_ORDER if p in include)
    return f"hddm_{dep}_{inc}"


def config_from_stem(stem: str):
    body = stem[5:] if stem.startswith("hddm_") else stem
    dep_part, inc_part = body.split("_", 1)
    depends_on = {p: CONDITION for p in dep_part}
    include = list(inc_part)
    return depends_on, include


MODEL_CANDIDATES = []
for stem in ALL_MODEL_STEMS:
    dep, inc = config_from_stem(stem)
    assert model_file_stem(dep, inc) == stem, stem
    MODEL_CANDIDATES.append({"stem": stem, "depends_on": dep, "include": inc})

depends_on, include = config_from_stem(ACTIVE_MODEL_STEM)
MODEL_FILE_STEM = model_file_stem(depends_on, include)
assert MODEL_FILE_STEM == ACTIVE_MODEL_STEM, (MODEL_FILE_STEM, ACTIVE_MODEL_STEM)

if TEST_RUN:
    OUTPUT_STEM = f"{MODEL_FILE_STEM}_test_{N_ITER}iter_{BURN}burn"
    MODEL_SAVE_DIR = os.path.join(MODEL_DIR, "_test_runs")
else:
    OUTPUT_STEM = MODEL_FILE_STEM
    MODEL_SAVE_DIR = MODEL_DIR

print(f"Active model: {MODEL_FILE_STEM}")
print(f"  depends_on = {depends_on}")
print(f"  include    = {include}")
print(f"Candidates: {len(MODEL_CANDIDATES)}")
if TEST_RUN:
    print(f"TEST_RUN=True → outputs under: {MODEL_SAVE_DIR}")
    print(f"  output stem: {OUTPUT_STEM}")
else:
    print(f"Production outputs → {MODEL_SAVE_DIR}")


Active model: hddm_va_vat
  depends_on = {'v': ['transition', 'prob', 'long_short'], 'a': ['transition', 'prob', 'long_short']}
  include    = ['v', 'a', 't']
Candidates: 25
TEST_RUN=True → outputs under: d:\文件同步\文件\数据分析结果\create_figures\hddm_figures\hddm_analysis\models\transition_prob_duration_10000samples\_test_runs
  output stem: hddm_va_vat_test_100iter_20burn


In [3]:
def check_kabuki_multichain_patch() -> None:
    spec = importlib.util.find_spec("kabuki.hierarchical")
    if spec is None or not spec.origin:
        raise RuntimeError("kabuki is not installed")
    text = Path(spec.origin).read_text(encoding="utf-8")
    if "def _sample_one_chain" not in text:
        raise RuntimeError(
            "Multi-chain sampling requires the kabuki patch. Run once:\n"
            "  python ../scripts/patch_kabuki_multichain.py"
        )


check_kabuki_multichain_patch()

if "model" in globals():
    del model  # avoid accidentally exporting a stale fit

model = hddm.HDDM(
    hddm_data,
    depends_on=depends_on,
    include=include,
    is_group_model=True,
    p_outlier=P_OUTLIER,
)
model.find_starting_values()

t0 = time.perf_counter()
model.sample(N_ITER, burn=BURN, chains=N_CHAINS, parallel=True)
SAMPLING_SECONDS = time.perf_counter() - t0
print(f"Sampling finished in {SAMPLING_SECONDS / 3600:.2f} h ({SAMPLING_SECONDS:.0f} s)")

No model attribute --> setting up standard HDDM
Set model to ddm


d:\ProgramData\envs\hddm\lib\site-packages\kabuki-0.6.5-py3.8.egg\kabuki\hierarchical.py:172: FutureWarning: In a future version, object-dtype columns with all-bool values will not be included in reductions with bool_only=True. Explicitly cast to bool dtype instead.
  self.nodes_db = pd.concat([self.nodes_db, row])
d:\ProgramData\envs\hddm\lib\site-packages\kabuki-0.6.5-py3.8.egg\kabuki\hierarchical.py:172: FutureWarning: In a future version, object-dtype columns with all-bool values will not be included in reductions with bool_only=True. Explicitly cast to bool dtype instead.
  self.nodes_db = pd.concat([self.nodes_db, row])
d:\ProgramData\envs\hddm\lib\site-packages\kabuki-0.6.5-py3.8.egg\kabuki\hierarchical.py:172: FutureWarning: In a future version, object-dtype columns with all-bool values will not be included in reductions with bool_only=True. Explicitly cast to bool dtype instead.
  self.nodes_db = pd.concat([self.nodes_db, row])
d:\ProgramData\envs\hddm\lib\site-packages\kabuki

Sampling finished in 0.02 h (64 s)


In [4]:
def safe_print_stats(fitted_model, print_hidden=False):
    """print_stats() needs deviance traces for all chains (parallel patch merges them)."""
    stats = fitted_model.gen_stats(print_hidden=print_hidden)
    print(stats.to_string())
    try:
        info = fitted_model.dic_info
        print(f"\nDIC: {info['DIC']:.4f}")
        print(f"deviance: {info['deviance']:.4f}")
        print(f"pD: {info['pD']:.4f}")
    except (KeyError, Exception) as exc:
        print(f"\n(DIC unavailable: {exc})")
        print("If using parallel multi-chain sampling, re-run:")
        print("  python ../scripts/patch_kabuki_multichain.py")
        print("Then restart the kernel and re-sample.")


safe_print_stats(model)

KeyError: 3

## Export posterior summaries

The next cell defines helper functions. The final cell runs LOO / WAIC (+ optional PPC) and writes CSV + `.nc` outputs.

Note: `.nc` stores posterior arrays only — not a reloadable HDDM model object. If PPC was not saved, re-run the export cell or call `post_pred_gen` again.

In [ ]:
PCT_LEVELS = [10, 30, 50, 70, 90]
N_PPC_SAMPLES = 100
# Thin posterior draws when evaluating pointwise log-likelihood for LOO/WAIC
# (full 4×(N_ITER−BURN) can be slow; 500 draws is usually enough for ranking).
N_IC_DRAWS = int(os.environ.get("HDDM_IC_DRAWS", "500"))


def extract_posterior_dict(fitted_model, n_chains: int, n_draws: int) -> dict:
    """Stack all MCMC chains (get_traces() returns the last chain only)."""
    posterior_dict = {}
    for stoch in fitted_model.get_stochastics().node:
        if not hasattr(stoch, "trace") or stoch.trace is None:
            continue
        name = stoch.__name__
        trace_vals = np.asarray(stoch.trace.gettrace(chain=None))
        expected = n_chains * n_draws
        if trace_vals.size != expected:
            raise ValueError(
                f"{name}: expected {expected} draws, got {trace_vals.size}. "
                "Check N_CHAINS / N_ITER / BURN match the sampling cell."
            )
        posterior_dict[name] = trace_vals.reshape(n_chains, n_draws)
    if not posterior_dict:
        raise RuntimeError(
            "No posterior traces found. Run the sampling cell successfully before export."
        )
    return posterior_dict


def group_level_posterior_only(posterior_dict: dict) -> dict:
    """Remove subject-level random-effect nodes before writing a shareable .nc."""
    return {k: v for k, v in posterior_dict.items() if "_subj" not in k}


def compute_dic(fitted_model) -> float:
    """Compute DIC without PyMC's chain=-1 lookup (breaks after parallel merge)."""
    try:
        mc = fitted_model.mc
        dev_trace = mc.db.trace("deviance")
        if not dev_trace._trace:
            raise KeyError("deviance trace is empty")
        all_dev = np.concatenate(
            [np.asarray(dev_trace._trace[k]) for k in sorted(dev_trace._trace.keys())]
        )
        mean_deviance = np.mean(all_dev, axis=0)
        for stochastic in mc.stochastics:
            try:
                stoch_trace = mc.db.trace(stochastic.__name__)
                if not stoch_trace._trace:
                    continue
                all_vals = np.concatenate(
                    [np.asarray(stoch_trace._trace[k]) for k in sorted(stoch_trace._trace.keys())]
                )
                stochastic.value = np.mean(all_vals, axis=0)
            except (KeyError, TypeError):
                pass
        return float(2 * mean_deviance - mc.deviance)
    except Exception as exc:
        print(f"Warning: could not compute DIC — {exc}")
        print("If using parallel multi-chain sampling, re-run:")
        print("  python ../scripts/patch_kabuki_multichain.py")
        print("Then restart the kernel and re-sample.")
    return float("nan")


def _pointwise_loglik_from_observeds(fitted_model, n_chains: int, n_draws: int, n_ic_draws: int):
    """Evaluate observed-node logp on a thinned posterior for ArviZ LOO/WAIC.

    Each HDDM observed node pools trials for one subject×condition cell, so
    "pointwise" here is leave-one-node-out (standard for kabuki/HDDM ICs).
    """
    observeds = list(fitted_model.get_observeds()["node"])
    stochastics = [s for s in fitted_model.get_stochastics().node if hasattr(s, "trace") and s.trace is not None]
    if not observeds or not stochastics:
        raise RuntimeError("No observed/stochastic nodes available for LOO/WAIC.")

    traces = {}
    for stoch in stochastics:
        vals = np.asarray(stoch.trace.gettrace(chain=None)).reshape(n_chains, n_draws)
        traces[stoch.__name__] = vals

    # Evenly spaced draw indices across the stacked posterior
    total = n_chains * n_draws
    n_keep = int(min(max(n_ic_draws, 50), total))
    flat_idx = np.linspace(0, total - 1, n_keep, dtype=int)
    chain_idx = flat_idx // n_draws
    draw_idx = flat_idx % n_draws
    # Reshape as a single pseudo-chain for ArviZ (n_keep draws)
    n_obs = len(observeds)
    loglik = np.empty((1, n_keep, n_obs), dtype=float)

    for j, (c, d) in enumerate(zip(chain_idx, draw_idx)):
        for stoch in stochastics:
            stoch.value = traces[stoch.__name__][c, d]
        for i, node in enumerate(observeds):
            lp = float(node.logp)
            # Guard against -inf / NaN from numerical underflow
            if not np.isfinite(lp):
                lp = -700.0
            loglik[0, j, i] = lp
    return loglik


def compute_loo_waic(fitted_model, n_chains: int, n_draws: int, n_ic_draws: int = N_IC_DRAWS):
    """Return (loo_ic, waic_ic) on the deviance scale (lower is better)."""
    loo_ic = float("nan")
    waic_ic = float("nan")
    try:
        loglik = _pointwise_loglik_from_observeds(
            fitted_model, n_chains, n_draws, n_ic_draws
        )
        idata_ll = az.from_dict(
            log_likelihood={"obs": loglik},
            coords={
                "chain": [0],
                "draw": list(range(loglik.shape[1])),
                "obs_id": list(range(loglik.shape[2])),
            },
            dims={"obs": ["chain", "draw", "obs_id"]},
        )
        loo_res = az.loo(idata_ll, pointwise=False, scale="deviance")
        waic_res = az.waic(idata_ll, pointwise=False, scale="deviance")
        # ArviZ ELPDData: attribute names vary slightly by version
        loo_ic = float(getattr(loo_res, "loo", getattr(loo_res, "elpd_loo", np.nan)))
        waic_ic = float(getattr(waic_res, "waic", getattr(waic_res, "elpd_waic", np.nan)))
    except Exception as exc:
        print(f"Warning: could not compute LOO/WAIC — {exc}")
    return loo_ic, waic_ic


def generate_ppc(fitted_model, n_ppc: int, n_chains: int):
    """Posterior predictive check; returns posterior_predictive dict for InferenceData."""
    posterior_predictive_dict = {}
    try:
        ppc_data = hddm.utils.post_pred_gen(fitted_model, samples=n_ppc)
    except Exception as exc:
        print(f"Warning: PPC generation failed — {exc}")
        return posterior_predictive_dict

    if not isinstance(ppc_data, pd.DataFrame) or "rt" not in ppc_data.columns:
        print(f"Warning: unexpected PPC format ({type(ppc_data)})")
        return posterior_predictive_dict

    rt_simulated = ppc_data["rt"].values
    if len(rt_simulated) > 0 and hasattr(rt_simulated[0], "__len__") and not isinstance(rt_simulated[0], str):
        rt_simulated = np.array(
            [np.array(rt) if hasattr(rt, "__len__") else [rt] for rt in rt_simulated]
        )
    rt_simulated_flat = np.asarray(rt_simulated).flatten()

    n_obs = None
    try:
        n_obs = int(len(fitted_model.data))
    except Exception:
        n_obs = 0

    n_ppc_actual = rt_simulated.shape[0] if hasattr(rt_simulated, "shape") and len(rt_simulated.shape) >= 1 else len(rt_simulated)

    if hasattr(rt_simulated, "shape") and len(rt_simulated.shape) == 2:
        chains_per_ppc = min(n_chains, n_ppc_actual)
        draws_per_chain = n_ppc_actual // chains_per_ppc
        if draws_per_chain > 0:
            total = chains_per_ppc * draws_per_chain
            posterior_predictive_dict["rt"] = rt_simulated[:total].reshape(
                chains_per_ppc, draws_per_chain, rt_simulated.shape[1]
            )
    elif rt_simulated_flat.size > 0 and n_obs > 0 and rt_simulated_flat.size % n_obs == 0:
        n_draws_ppc = rt_simulated_flat.size // n_obs
        chains_per_ppc = min(n_chains, n_draws_ppc)
        draws_per_chain = n_draws_ppc // chains_per_ppc
        if draws_per_chain > 0:
            total = chains_per_ppc * draws_per_chain * n_obs
            posterior_predictive_dict["rt"] = rt_simulated_flat[:total].reshape(
                chains_per_ppc, draws_per_chain, n_obs
            )

    if "response" in ppc_data.columns:
        response_simulated = ppc_data["response"].values
        if len(response_simulated) > 0:
            if hasattr(response_simulated[0], "__len__") and not isinstance(response_simulated[0], str):
                response_simulated = np.array(
                    [np.array(r) if hasattr(r, "__len__") else [r] for r in response_simulated]
                )
            n_ppc_actual = len(response_simulated)
            chains_per_ppc = min(n_chains, n_ppc_actual)
            draws_per_chain = n_ppc_actual // chains_per_ppc
            if draws_per_chain > 0 and hasattr(response_simulated, "shape") and len(response_simulated.shape) == 2:
                total = chains_per_ppc * draws_per_chain
                posterior_predictive_dict["response"] = response_simulated[:total].reshape(
                    chains_per_ppc, draws_per_chain, response_simulated.shape[1]
                )

    return posterior_predictive_dict


def build_inference_data(posterior_dict, posterior_predictive_dict, n_chains, n_draws):
    idata = az.from_dict(
        posterior=posterior_dict,
        coords={"chain": range(n_chains), "draw": range(n_draws)},
        dims={param: ["chain", "draw"] for param in posterior_dict},
    )
    if not posterior_predictive_dict:
        return idata
    ref = posterior_predictive_dict[next(iter(posterior_predictive_dict))]
    ppc_coords = {
        "chain": range(ref.shape[0]),
        "draw": range(ref.shape[1]),
        "obs": range(ref.shape[2]),
    }
    ppc_dims = {var: ["chain", "draw", "obs"] for var in posterior_predictive_dict}
    ppc_idata = az.from_dict(
        posterior_predictive=posterior_predictive_dict,
        coords=ppc_coords,
        dims=ppc_dims,
    )
    return az.concat([idata, ppc_idata], dim=None)


def print_convergence_summary(idata) -> None:
    summary = az.summary(idata, round_to=3)
    if "r_hat" not in summary.columns:
        print("R-hat not available in summary.")
        return
    bad = summary[summary["r_hat"] > 1.01]
    print(f"Parameters with R-hat > 1.01: {len(bad)} / {len(summary)}")
    if len(bad):
        display(bad[["mean", "sd", "r_hat", "ess_bulk"]].head(20))


In [ ]:
if "model" not in globals():
    raise RuntimeError("Run the sampling cell first — `model` is not defined.")

n_chains = N_CHAINS
n_draws = N_ITER - BURN

print("Extracting posterior traces …")
posterior_full = extract_posterior_dict(model, n_chains, n_draws)
n_subject_nodes = sum(1 for k in posterior_full if "_subj" in k)
print(f"  total nodes: {len(posterior_full)} (subject-level: {n_subject_nodes})")

posterior_dict = (
    group_level_posterior_only(posterior_full)
    if EXPORT_GROUP_LEVEL_ONLY
    else posterior_full
)
if EXPORT_GROUP_LEVEL_ONLY:
    print(f"  group-level nodes kept: {len(posterior_dict)}")

print("Computing DIC …")
dic_value = compute_dic(model)
print(f"  DIC = {dic_value:.2f}" if np.isfinite(dic_value) else "  DIC = unavailable")

print(f"Computing LOO / WAIC (thinned to {N_IC_DRAWS} draws) …")
loo_value, waic_value = compute_loo_waic(model, n_chains, n_draws, N_IC_DRAWS)
print(f"  LOO  (deviance) = {loo_value:.2f}" if np.isfinite(loo_value) else "  LOO = unavailable")
print(f"  WAIC (deviance) = {waic_value:.2f}" if np.isfinite(waic_value) else "  WAIC = unavailable")

print(f"Generating PPC ({N_PPC_SAMPLES} draws) …")
posterior_predictive_dict = generate_ppc(model, N_PPC_SAMPLES, n_chains)

idata = build_inference_data(posterior_dict, posterior_predictive_dict, n_chains, n_draws)
print_convergence_summary(idata)

os.makedirs(MODEL_SAVE_DIR, exist_ok=True)

pickle_file = os.path.join(MODEL_SAVE_DIR, f"{OUTPUT_STEM}.pickle")
with open(pickle_file, "wb") as f:
    pickle.dump(model, f, protocol=4)
print(f"Saved model pickle: {pickle_file}")

output_file = os.path.join(MODEL_SAVE_DIR, f"{OUTPUT_STEM}_combined.nc")
az.to_netcdf(idata, output_file)
print(f"Saved InferenceData: {output_file}")

model_info = pd.DataFrame(
    {
        "model_name": [MODEL_FILE_STEM],
        "output_stem": [OUTPUT_STEM],
        "test_run": [TEST_RUN],
        "dic": [dic_value],
        "loo": [loo_value],
        "waic": [waic_value],
        "n_parameters": [len(posterior_dict)],
        "n_samples": [n_chains * n_draws],
        "n_chains": [n_chains],
        "n_iter": [N_ITER],
        "burn": [BURN],
        "p_outlier": [P_OUTLIER],
        "min_rt_ms": [MIN_RT_MS],
        "sampling_seconds": [round(globals().get("SAMPLING_SECONDS", float("nan")), 1)],
        "export_group_level_only": [EXPORT_GROUP_LEVEL_ONLY],
    }
)
model_info_file = os.path.join(MODEL_SAVE_DIR, f"{OUTPUT_STEM}_model_info.csv")
model_info.to_csv(model_info_file, index=False, encoding="utf-8-sig")
print(f"Saved model info: {model_info_file}")

dic_loo_waic_file = os.path.join(MODEL_SAVE_DIR, f"{OUTPUT_STEM}_dic_loo_waic.csv")
pd.DataFrame(
    {
        "model_name": [MODEL_FILE_STEM],
        "output_stem": [OUTPUT_STEM],
        "dic": [dic_value],
        "loo": [loo_value],
        "waic": [waic_value],
    }
).to_csv(dic_loo_waic_file, index=False, encoding="utf-8-sig")
print(f"Saved DIC/LOO/WAIC: {dic_loo_waic_file}")

print("\nExport complete.")
